In [1]:
# Core libraries
import pandas as pd
import numpy as np


# Machine Learning
from sklearn.preprocessing import StandardScaler

from sklearn.cluster import DBSCAN

# Statistics
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint
from statsmodels.tsa.stattools import adfuller

# Utilities
from itertools import combinations
import warnings

# Variables
import config as cfg

# Ignore warnings (useful for statistical tests)
warnings.filterwarnings("ignore")

In [ ]:
import importlib
import core.analyse as an

importlib.reload(an)

<module 'core.analyse' from 'c:\\Users\\Asus\\Desktop\\Git\\master-projects\\financial-market-analysis\\src\\core\\analyse.py'>

In [ ]:
from utils.io import get_stock_data

prices = get_stock_data(cfg.TICKERS)

print("All data downloaded.")

[*********************100%***********************]  501 of 501 completed


All data downloaded.


In [ ]:
# Célula 3
prices = prices.dropna()

# DIVISÃO SOTA: Prevenir Look-Ahead Bias
# Usamos 2016-2023 para encontrar a estrutura dos pares (Formation)
# Usamos 2024-2026 para os Agentes operarem (Trading/Testing)
SPLIT_DATE = '2023-12-31'

prices_formation = prices.loc[:SPLIT_DATE]
prices_trading = prices.loc[SPLIT_DATE:]

print(f"Período de Formação (PCA/Cointegração): {prices_formation.shape[0]} dias")
print(f"Período de Trading (Agentes MAS): {prices_trading.shape[0]} dias")

Ticker,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WY,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2025-10-27,146.328461,268.298615,226.209961,129.070007,126.539017,85.940002,250.770004,357.799988,241.398865,60.962898,...,23.589035,124.907814,80.071976,114.173706,148.475266,80.150002,141.653229,102.846626,310.570007,146.392990
2025-10-28,145.589798,268.488251,225.723877,128.009995,125.971886,84.720001,253.350006,359.910004,237.763153,60.401886,...,23.314058,120.337288,79.208641,113.277565,149.568451,80.180000,139.780777,100.871872,274.309998,144.289337
2025-10-29,142.944519,269.186920,223.343033,126.480003,123.802872,85.849998,247.750000,337.859985,233.481720,59.565296,...,22.783749,120.357208,79.079636,114.675934,152.003296,76.510002,137.898407,99.445663,270.769989,142.830658
2025-10-30,143.104233,270.883667,226.378616,126.339996,124.041664,86.639999,249.250000,339.239990,231.355896,59.732616,...,23.088186,119.261864,80.965080,112.942749,150.592087,73.919998,137.769608,99.296059,261.369995,142.989441
2025-10-31,146.098877,269.855652,216.299698,126.540001,122.996956,86.309998,250.100006,340.309998,232.577759,59.575138,...,22.587336,118.485176,80.548294,112.617775,149.916306,75.940002,136.927521,100.293411,269.250000,143.477341


In [ ]:
# Célula 4
# Calculamos os retornos APENAS nos dados de Formação
returns_formation = prices_formation.pct_change().dropna()
returns_formation.head()

In [ ]:
# Célula 5
from itertools import combinations

# 1. Filtro de Ruído Institucional (Marchenko-Pastur)
# Esta função já faz o scale e retorna o PCA limpo!
X_pca_sota = an.calculate_RMT(returns_formation)

# 2. Descobrir o EPS ideal para o DBSCAN
# Olha para o gráfico que isto vai gerar. Onde a curva subir drasticamente (o cotovelo), é o teu EPS.
an.calculate_nearest_neighbors(X_pca_sota)

# 3. Aplicar o DBSCAN com o EPS ótimo (ajusta este valor baseado no gráfico acima, ex: 3.5 ou 4.0)
BEST_EPS = 3.5
pairs_grouped, clusters_dict = an.RMT_clustering(BEST_EPS, X_pca_sota, returns_formation)

# 4. Extrair todas as combinações (2 a 2) dentro dos clusters encontrados
candidate_pairs = []
for cluster_list in clusters_dict.values():
    if len(cluster_list) > 1:
        for combo in combinations(cluster_list, 2):
            candidate_pairs.append(list(combo))

print(f"\nPares Candidatos Extraídos: {len(candidate_pairs)}")

Ticker,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WY,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2025-10-28,-0.005048,0.000707,-0.002149,-0.008213,-0.004482,-0.014196,0.010288,0.005897,-0.015061,-0.009203,...,-0.011657,-0.036591,-0.010782,-0.007849,0.007363,0.000374,-0.013219,-0.019201,-0.116753,-0.014370
2025-10-29,-0.018169,0.002602,-0.010548,-0.011952,-0.017218,0.013338,-0.022104,-0.061265,-0.018007,-0.013850,...,-0.022746,0.000166,-0.001629,0.012345,0.016279,-0.045772,-0.013467,-0.014139,-0.012905,-0.010109
2025-10-30,0.001117,0.006303,0.013592,-0.001107,0.001929,0.009202,0.006054,0.004085,-0.009105,0.002809,...,0.013362,-0.009101,0.023842,-0.015114,-0.009284,-0.033852,-0.000934,-0.001504,-0.034716,0.001112
2025-10-31,0.020926,-0.003795,-0.044522,0.001583,-0.008422,-0.003809,0.003410,0.003154,0.005281,-0.002636,...,-0.021693,-0.006512,-0.005148,-0.002877,-0.004487,0.027327,-0.006112,0.010044,0.030149,0.003412
2025-11-03,-0.011410,-0.004882,-0.027885,0.002055,0.001780,-0.003592,-0.006637,-0.008345,-0.002221,-0.007269,...,-0.021304,0.056643,0.001109,-0.005247,-0.011667,-0.021596,0.008465,-0.006762,0.004940,0.001805


In [ ]:
# Célula 8
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

log_prices_form = np.log(prices_formation)
log_prices_full = np.log(prices) # Para o bot usar depois

valid_pairs = []
spreads_dict = {}

print(f"A testar {len(candidate_pairs)} pares candidatos com Augmented Dickey-Fuller...")

for p in candidate_pairs:
    s1, s2 = p[0], p[1]
    
    # Treinar o Beta SÓ nos dados de formação (Passado)
    y_form = log_prices_form[s1]
    x_form = log_prices_form[s2]
    
    x_with_const = sm.add_constant(x_form)
    ols_result = sm.OLS(y_form, x_with_const).fit()
    beta = ols_result.params[s2]
    
    # Calcular o Spread de formação para testar a cointegração
    spread_form = y_form - beta * x_form
    
    # Teste ADF na formação
    adf_result = adfuller(spread_form)
    p_value = adf_result[1]
    
    if p_value < 0.05:
        valid_pairs.append({'ativo_y': s1, 'ativo_x': s2, 'beta': beta, 'p_value': p_value})
        
        # Se é válido, criamos o Spread REAL para toda a linha do tempo (usando o beta passado)
        # É ESTE SPREAD que vai alimentar os Agentes de 2024 a 2026!
        spread_full = log_prices_full[s1] - beta * log_prices_full[s2]
        spreads_dict[f"{s1}_{s2}"] = spread_full

valid_pairs_df = pd.DataFrame(valid_pairs).sort_values(by='p_value')
print(f"-> Sobreviveram {len(valid_pairs_df)} pares fortemente cointegrados estruturalmente!")
valid_pairs_df.head(3)